# Temporal Mining Analysis — 2021 to 2025
Runs EfficientNet-B0 (`best_model.pth`) across all JP2 files year-by-year.
Outputs:
- Mining patch count and estimated area (km²) per year
- Expansion bar + line chart
- Year-over-year change heatmap
- Side-by-side overlay maps (one per year, best scene selected)
- CSV report

## Cell 1 — Imports

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models, transforms
import rasterio
from rasterio.enums import Resampling

print('Imports OK')

## Cell 2 — Config

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT   = r'R:\ai project image\data image big'   # folder containing 2021,2022... subfolders
MODEL_PATH  = 'saved_models/best_model.pth'           # EfficientNet-B0
OUTPUT_DIR  = 'inference_output/temporal'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model settings ────────────────────────────────────────────────────────────
PATCH_SIZE   = 224       # EfficientNet-B0 input
STRIDE       = 112       # 50% overlap — better coverage
BATCH_SIZE   = 32
THRESHOLD    = 0.5       # mining probability threshold
NUM_CLASSES  = 2

# Sentinel-2 10m resolution → each 64px patch = 640m x 640m
# For 224px patches at 10m/px → each patch = 2240m x 2240m = 5.0176 km²
PATCH_AREA_KM2 = (PATCH_SIZE * 10 / 1000) ** 2

# ── Year folders ─────────────────────────────────────────────────────────────
# Maps folder name → year label
YEAR_MAP = {
    'big 2021': 2021,
    '2022'    : 2022,
    '2023'    : 2023,
    '2024'    : 2024,
    '2025'    : 2025,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Patch area: {PATCH_AREA_KM2:.4f} km²')
print(f'Output dir: {OUTPUT_DIR}')

## Cell 3 — Load Model

In [ ]:
def build_efficientnet_b0(model_path):
    model = models.efficientnet_b0(weights=None)
    in_f  = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_f, 256), nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(256, NUM_CLASSES)
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()
    return model

model = build_efficientnet_b0(MODEL_PATH)
print(f'Model loaded from {MODEL_PATH}')

tf = transforms.Compose([
    transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

## Cell 4 — Inference Helper

In [ ]:
def load_jp2_as_rgb(jp2_path, max_dim=4096):
    """Load a Sentinel-2 TCI JP2 and downsample if too large."""
    with rasterio.open(jp2_path) as src:
        h, w = src.height, src.width
        scale = min(1.0, max_dim / max(h, w))
        new_h, new_w = int(h * scale), int(w * scale)
        data = src.read(
            out_shape=(3, new_h, new_w),
            resampling=Resampling.bilinear
        )
    # data shape: (3, H, W) — convert to uint8 RGB
    data = np.moveaxis(data, 0, -1)          # (H, W, 3)
    if data.dtype != np.uint8:
        data = (data / data.max() * 255).astype(np.uint8)
    return data


def run_inference(img_rgb, stride=STRIDE):
    """
    Slide a window across the image.
    Returns:
      mining_mask  : bool array (H, W) — True where mining detected
      prob_map     : float array (H, W) — mining probability per pixel
      n_mining     : number of mining patches
      n_total      : total patches processed
    """
    H, W, _ = img_rgb.shape
    mining_mask = np.zeros((H, W), dtype=np.float32)
    count_map   = np.zeros((H, W), dtype=np.float32)

    patches, coords = [], []
    for y in range(0, H - PATCH_SIZE + 1, stride):
        for x in range(0, W - PATCH_SIZE + 1, stride):
            patch = img_rgb[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            patches.append(tf(Image.fromarray(patch)))
            coords.append((y, x))

    if not patches:
        return mining_mask.astype(bool), mining_mask, 0, 0

    # Batch inference
    all_probs = []
    for i in range(0, len(patches), BATCH_SIZE):
        batch = torch.stack(patches[i:i+BATCH_SIZE]).to(device)
        with torch.no_grad():
            out   = model(batch)
            probs = torch.softmax(out, dim=1)[:, 0].cpu().numpy()  # class 0 = mining
        all_probs.extend(probs)

    n_mining = 0
    for prob, (y, x) in zip(all_probs, coords):
        mining_mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += prob
        count_map[y:y+PATCH_SIZE,   x:x+PATCH_SIZE] += 1
        if prob >= THRESHOLD:
            n_mining += 1

    count_map[count_map == 0] = 1
    prob_map = mining_mask / count_map
    return prob_map >= THRESHOLD, prob_map, n_mining, len(patches)


print('Inference helpers defined.')

## Cell 5 — Run Inference Across All Years

In [ ]:
results = []   # list of dicts, one per JP2 file

for folder_name, year in sorted(YEAR_MAP.items(), key=lambda x: x[1]):
    folder_path = os.path.join(DATA_ROOT, folder_name)
    jp2_files   = sorted(glob.glob(os.path.join(folder_path, '*.jp2')))

    if not jp2_files:
        print(f'{year}: no JP2 files found in {folder_path}')
        continue

    print(f'\n── {year} ({len(jp2_files)} files) ──────────────────────')

    for jp2_path in jp2_files:
        fname = os.path.basename(jp2_path)
        # Extract date from filename e.g. T45QUE_20230107T045201_TCI_10m
        date_match = re.search(r'_(\d{8})T', fname)
        date_str   = date_match.group(1) if date_match else 'unknown'

        print(f'  Processing {fname} ...', end=' ')
        try:
            img_rgb = load_jp2_as_rgb(jp2_path)
            mining_mask, prob_map, n_mining, n_total = run_inference(img_rgb)
            area_km2 = n_mining * PATCH_AREA_KM2
            pct      = 100 * n_mining / n_total if n_total > 0 else 0

            print(f'{n_mining}/{n_total} patches mining  |  {area_km2:.2f} km²  ({pct:.1f}%)')

            results.append({
                'year'       : year,
                'date'       : date_str,
                'filename'   : fname,
                'n_mining'   : n_mining,
                'n_total'    : n_total,
                'area_km2'   : area_km2,
                'pct_mining' : pct,
                'img_rgb'    : img_rgb,      # keep for maps
                'prob_map'   : prob_map,
                'mining_mask': mining_mask,
            })
        except Exception as e:
            print(f'ERROR: {e}')

print(f'\nTotal scenes processed: {len(results)}')

## Cell 6 — Aggregate Per Year (best scene = max mining area)

In [ ]:
df_all = pd.DataFrame([{k:v for k,v in r.items()
                         if k not in ('img_rgb','prob_map','mining_mask')}
                        for r in results])

# Per year: take the scene with maximum detected mining area
# (most cloud-free scene will show the most mining)
df_year = (df_all.sort_values('area_km2', ascending=False)
                 .groupby('year', as_index=False)
                 .first()
                 .sort_values('year'))

# Year-over-year change
df_year['change_km2'] = df_year['area_km2'].diff()
df_year['change_pct'] = df_year['area_km2'].pct_change() * 100

print('\n' + '='*60)
print('  TEMPORAL ANALYSIS — YEAR-WISE SUMMARY')
print('='*60)
print(f'{"Year":<6} {"Best date":<10} {"Mining patches":<16} {"Area (km²)":<12} {"Change"}')
print('-'*60)
for _, row in df_year.iterrows():
    chg = f"{row['change_km2']:+.2f} km² ({row['change_pct']:+.1f}%)" if not pd.isna(row['change_km2']) else 'baseline'
    print(f"{int(row['year']):<6} {row['date']:<10} {int(row['n_mining']):<16} {row['area_km2']:<12.2f} {chg}")
print('='*60)

# Save CSV
csv_path = os.path.join(OUTPUT_DIR, 'temporal_summary.csv')
df_year.drop(columns=['img_rgb','prob_map','mining_mask'], errors='ignore').to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

## Cell 7 — Expansion Chart (bar + line)

In [ ]:
years     = df_year['year'].astype(int).tolist()
areas     = df_year['area_km2'].tolist()
changes   = df_year['change_km2'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: bar + line overlay ──────────────────────────────────────────────────
ax1 = axes[0]
bar_colors = ['#c0392b' if a == max(areas) else '#2980b9' for a in areas]
bars = ax1.bar(years, areas, color=bar_colors, alpha=0.85, width=0.5, zorder=2)
ax1.plot(years, areas, 'o-', color='#e74c3c', linewidth=2, markersize=7, zorder=3)
for bar, area in zip(bars, areas):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{area:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Estimated Mining Area (km²)', fontsize=12)
ax1.set_title('Mining Area Expansion 2021–2025', fontsize=13)
ax1.set_xticks(years)
ax1.set_ylim(0, max(areas) * 1.2)
ax1.grid(axis='y', alpha=0.3)
peak = mpatches.Patch(color='#c0392b', alpha=0.85, label='Peak year')
rest = mpatches.Patch(color='#2980b9', alpha=0.85, label='Other years')
ax1.legend(handles=[peak, rest], fontsize=10)

# ── Right: YoY change bar ─────────────────────────────────────────────────────
ax2 = axes[1]
chg_vals   = [c for c in changes[1:]]   # skip first NaN
chg_years  = years[1:]
chg_colors = ['#27ae60' if c >= 0 else '#e74c3c' for c in chg_vals]
bars2 = ax2.bar(chg_years, chg_vals, color=chg_colors, alpha=0.85, width=0.5)
for bar, val in zip(bars2, chg_vals):
    ypos = val + 0.1 if val >= 0 else val - 0.4
    ax2.text(bar.get_x() + bar.get_width()/2, ypos,
             f'{val:+.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Change in Mining Area (km²)', fontsize=12)
ax2.set_title('Year-over-Year Change', fontsize=13)
ax2.set_xticks(chg_years)
ax2.grid(axis='y', alpha=0.3)
inc = mpatches.Patch(color='#27ae60', alpha=0.85, label='Increase')
dec = mpatches.Patch(color='#e74c3c', alpha=0.85, label='Decrease')
ax2.legend(handles=[inc, dec], fontsize=10)

plt.suptitle('Illegal Mining Temporal Analysis — T45QUE Tile', fontsize=14, y=1.02)
plt.tight_layout()
chart_path = os.path.join(OUTPUT_DIR, 'mining_expansion_chart.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {chart_path}')

## Cell 8 — Side-by-Side Overlay Maps (one per year)

In [ ]:
# For each year pick the best scene (highest mining area — same as Cell 6)
best_scenes = {}
for r in results:
    yr = r['year']
    if yr not in best_scenes or r['area_km2'] > best_scenes[yr]['area_km2']:
        best_scenes[yr] = r

sorted_years = sorted(best_scenes.keys())
n_years = len(sorted_years)

fig, axes = plt.subplots(1, n_years, figsize=(5 * n_years, 5))
if n_years == 1:
    axes = [axes]

for ax, year in zip(axes, sorted_years):
    scene = best_scenes[year]
    img   = scene['img_rgb']
    mask  = scene['mining_mask']

    # Create red overlay
    overlay = img.copy()
    overlay[mask, 0] = 220   # red channel up
    overlay[mask, 1] = (overlay[mask, 1] * 0.3).astype(np.uint8)
    overlay[mask, 2] = (overlay[mask, 2] * 0.3).astype(np.uint8)

    ax.imshow(overlay)
    ax.set_title(f'{year}\n{scene["area_km2"]:.1f} km²  ({scene["date"]})',
                 fontsize=11, fontweight='bold')
    ax.axis('off')

    # Mining area annotation
    ax.text(0.02, 0.03, f'Mining: {scene["n_mining"]} patches',
            transform=ax.transAxes, fontsize=8,
            color='white', bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.6))

# Red patch legend
red_patch = mpatches.Patch(color='red', alpha=0.7, label='Detected mining')
fig.legend(handles=[red_patch], loc='lower center', fontsize=11,
           bbox_to_anchor=(0.5, -0.04), ncol=1)

plt.suptitle('Mining Detection Overlay — Year by Year (T45QUE)', fontsize=14)
plt.tight_layout()
maps_path = os.path.join(OUTPUT_DIR, 'mining_overlay_maps.png')
plt.savefig(maps_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {maps_path}')

## Cell 9 — Heatmap: Mining Probability Across Years

In [ ]:
# Stack all best prob_maps and show average intensity per year
# Resize all to same shape for comparison
TARGET_H, TARGET_W = 512, 512

fig, axes = plt.subplots(1, n_years, figsize=(4 * n_years, 4))
if n_years == 1:
    axes = [axes]

for ax, year in zip(axes, sorted_years):
    scene    = best_scenes[year]
    prob_map = scene['prob_map']

    # Resize prob map
    from PIL import Image as PILImage
    prob_img = PILImage.fromarray((prob_map * 255).astype(np.uint8))
    prob_res = np.array(prob_img.resize((TARGET_W, TARGET_H), PILImage.BILINEAR)) / 255.0

    im = ax.imshow(prob_res, cmap='hot', vmin=0, vmax=1)
    ax.set_title(f'{year}', fontsize=12, fontweight='bold')
    ax.axis('off')

plt.colorbar(im, ax=axes[-1], fraction=0.046, pad=0.04, label='Mining probability')
plt.suptitle('Mining Probability Heatmap by Year', fontsize=14)
plt.tight_layout()
heat_path = os.path.join(OUTPUT_DIR, 'mining_heatmap_years.png')
plt.savefig(heat_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {heat_path}')

## Cell 10 — Full CSV Report (all scenes)

In [ ]:
df_full = df_all.copy()
df_full['date_fmt'] = pd.to_datetime(df_full['date'], format='%Y%m%d', errors='coerce')
df_full = df_full.sort_values(['year','date_fmt'])

full_csv = os.path.join(OUTPUT_DIR, 'all_scenes_report.csv')
df_full[['year','date','filename','n_mining','n_total','area_km2','pct_mining']].to_csv(full_csv, index=False)

print('\nAll scenes report:')
print(df_full[['year','date','n_mining','area_km2','pct_mining']].to_string(index=False))
print(f'\nSaved: {full_csv}')

print('\n' + '='*50)
print('  TEMPORAL ANALYSIS COMPLETE')
print('='*50)
print(f'Outputs in {OUTPUT_DIR}/')
for f in os.listdir(OUTPUT_DIR):
    print(f'  {f}')